# 00: Setup and sanity checks

Prerequisite checks for the run matrix (Cypher + AQL + Gremlin targets, Ollama + Anthropic models, LDBC):

1. Resolve repo root and import the `harness` package.
2. Check Ollama is reachable and the matrix's Ollama models are pulled (the only hard requirement for the DB-free notebooks 01-04).
3. Warn (not fail) if the execution databases are down (Postgres, Neo4j, ArangoDB) -- they are only needed for the deferred notebook 05.
4. Summarise the gold dataset(s) the run matrix will iterate over.

Run this first. If Ollama is down or a model is missing, fix that before `01_translation_run` (whose Anthropic cells need `ANTHROPIC_API_KEY`). For notebook 05 you also need the databases loaded with LDBC SF1.

In [1]:
from __future__ import annotations

import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "pyproject.toml").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / "eval"))

import json
import os
import urllib.error
import urllib.request

from harness import CACHE_DIR, METRICS_DIR, RECORDS_DIR, RUN_MATRIX, load_dataset

for d in (RECORDS_DIR, METRICS_DIR, CACHE_DIR):
    d.mkdir(parents=True, exist_ok=True)
print('Repo root:', REPO_ROOT)
print('Records:  ', RECORDS_DIR)
print('Run matrix:')
for rc in RUN_MATRIX:
    print(f'  - {rc.dataset} x {rc.target} x {rc.model} ({rc.provider}), validation={rc.validation_mode or "default"}')

Repo root: /Users/ivona.obonova/school/rows2graph/rows2graph
Outputs:   /Users/ivona.obonova/school/rows2graph/rows2graph/evaluation/outputs
Run matrix:
  - ldbc x cypher x qwen3-coder:30b (ollama), validation=default


## Ollama liveness

The library's Ollama client talks to `http://localhost:11434`. We confirm every Ollama-backed matrix cell's model is pulled.

In [2]:
def ollama_models(host):
    with urllib.request.urlopen(f'{host}/api/tags', timeout=5) as resp:
        data = json.loads(resp.read())
    return [m['name'] for m in data.get('models', [])]

ollama_cells = [rc for rc in RUN_MATRIX if rc.provider == 'ollama']
if ollama_cells:
    host = ollama_cells[0].host
    try:
        available = ollama_models(host)
        print(f'Ollama at {host}: {len(available)} model(s) available.')
        for rc in ollama_cells:
            ok = any(rc.model == m or m.startswith(rc.model) for m in available)
            print(f'  {rc.model}: ' + ('OK' if ok else 'MISSING -- run: ollama pull ' + rc.model))
    except (urllib.error.URLError, OSError) as exc:
        print(f'WARNING: could not reach Ollama at {host}: {exc}')
        print('Start it with `ollama serve` before running 01_translation_run.')
else:
    print('No Ollama cells in RUN_MATRIX; skipping Ollama check.')

Ollama at http://localhost:11434: 16 model(s) available.
  qwen3-coder:30b: OK


## Optional databases (deferred execution metrics)

Execution accuracy (notebook 05) runs the gold SQL on graphonauts2's Postgres and the generated query on the graph DB: Cypher on Neo4j, AQL on ArangoDB (db `graphonauts`), all loaded with LDBC SF1; Gremlin runs on the in-memory TinkerGraph, which is not checked here (it must run with Neo4j/ArangoDB stopped). That is deferred in this pass, so these checks only **warn**. The ArangoDB check also verifies the mapping-aligned unified edge collections exist (built by `eval/scripts/build_arango_unified_edges.py`); without them every AQL traversal query scores 0.

In [ ]:
from harness import execution as ex


def _warn_db(name, fn):
    try:
        fn()
        print(f'  {name}: reachable')
    except Exception as exc:
        detail = (str(exc).splitlines() or [''])[0][:80] or type(exc).__name__
        print(f'  {name}: unavailable ({type(exc).__name__}: {detail}) -- needed only for notebook 05')

# Endpoints come from harness.execution (shared with notebook 05 and
# eval/scripts/validate_gold.py), so the checks and the executors agree.
def _check_pg():
    import psycopg
    with psycopg.connect(ex.PG_DSN, connect_timeout=3) as c, c.cursor() as cur:
        cur.execute('SELECT 1')

def _check_neo4j():
    from neo4j import GraphDatabase
    drv = GraphDatabase.driver(ex.NEO4J_URI, auth=(ex.NEO4J_USER, os.environ.get('NEO4J_PASSWORD', '')))
    try:
        drv.verify_connectivity()
    finally:
        drv.close()

def _check_arango():
    from arango import ArangoClient
    client = ArangoClient(hosts=ex.ARANGO_URL)
    db = client.db(ex.ARANGO_DB, username=ex.ARANGO_USER, password=os.environ.get('ARANGO_PASSWORD', ''))
    db.properties()  # round-trips to the server; raises if unreachable / auth-bad
    if not db.has_collection('KNOWS'):
        raise RuntimeError('unified edges missing -- run eval/scripts/build_arango_unified_edges.py')

print('Optional databases (deferred execution metrics):')
_warn_db('Postgres (5433)', _check_pg)
_warn_db('Neo4j (7687)', _check_neo4j)
_warn_db('ArangoDB (8529, db graphonauts)', _check_arango)


## Gold dataset summary

What the run matrix will iterate over.

In [4]:
import pandas as pd

datasets = sorted({rc.dataset for rc in RUN_MATRIX})
rows = []
for ds in datasets:
    rows.extend(
        {'dataset': ds, 'id': q.id, 'difficulty': q.difficulty,
         'sql_features': ', '.join(q.sql_features) or '(none)',
         'targets': ', '.join(sorted(q.expected))}
        for q in load_dataset(ds)
    )
summary = pd.DataFrame(rows)
print(f'Total gold queries across matrix datasets: {len(summary)}')
display(summary.pivot_table(index='dataset', columns='difficulty', values='id', aggfunc='count', fill_value=0))
summary

Total gold queries across matrix datasets: 14


difficulty,easy,hard,medium
dataset,,,
ldbc,3,7,4


,dataset,id,difficulty,sql_features,targets
0,ldbc,ldbc_q01,easy,(none),"aql, cypher"
1,ldbc,ldbc_q02,easy,(none),"aql, cypher"
2,ldbc,ldbc_q03,easy,(none),"aql, cypher"
3,ldbc,ldbc_q04,hard,"aggregation, join, order_limit","aql, cypher"
4,ldbc,ldbc_q05,hard,"aggregation, join, order_limit, subquery, union","aql, cypher"
5,ldbc,ldbc_q06,medium,join,"aql, cypher"
6,ldbc,ldbc_q07,medium,join,"aql, cypher"
7,ldbc,ldbc_q08,hard,"distinct, join","aql, cypher"
8,ldbc,ldbc_q10,medium,"join, order_limit","aql, cypher"
9,ldbc,ldbc_q11,hard,"aggregation, join, order_limit","aql, cypher"


Checks done. Continue to `01_translation_run.ipynb`.